# Milestone 1: Data Profiling

**Nama:** Afzal Raditya Dharma  
**NIM:** 202310370311236

Profil awal dataset **APBD Pemda Indonesia (2011–2026)** menggunakan Polars LazyFrame
 dan DuckDB. Sumber serta deskripsi dataset tersedia di `data/README.md`.
Tujuan profiling adalah memeriksa ukuran, jumlah baris, tipe data, kelengkapan,
serta indikasi anomali sebelum cleaning pada Milestone 2.

In [1]:
from pathlib import Path
import duckdb
import polars as pl
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / 'data/raw/apbd_indonesia_2011_2026.parquet').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Dataset tidak ditemukan. Unduh apbd_indonesia_2011_2026.parquet ke data/raw/.'
    )
DATA_PATH = PROJECT_ROOT / 'data/raw/apbd_indonesia_2011_2026.parquet'
file_size_mb = DATA_PATH.stat().st_size / 1_000_000
print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {file_size_mb:.2f} MB')

Dataset: apbd_indonesia_2011_2026.parquet
Ukuran file: 14.24 MB


## Profil Polars

`scan_parquet()` membangun rencana query secara lazy. Hasil query dieksekusi saat
`collect()` dipanggil. Ringkasan mencakup jumlah baris dan jumlah null setiap kolom.
Dataset memenuhi batas tugas apabila berukuran minimal 500 MB **atau** memiliki
lebih dari 1 juta baris.

In [2]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())
ringkasan = lazy_data.select(
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
).collect()
display(ringkasan)
total_rows = ringkasan['jumlah_baris'][0]
memenuhi_kriteria = file_size_mb >= 500 or total_rows > 1_000_000
print(f'Jumlah baris: {total_rows:,}')
print(f'Jumlah kolom: {len(lazy_data.collect_schema())}')
print('Status:', 'MEMENUHI KRITERIA TUGAS' if memenuhi_kriteria else 'BELUM MEMENUHI KRITERIA TUGAS')

Schema({'tahun': Int16, 'bulan': Int8, 'id_provinsi': String, 'provinsi': String, 'id_pemda': String, 'pemda': String, 'no_akun': Int16, 'akun': String, 'anggaran': Float64, 'realisasi': Float64, 'persentase': Float64})
Jumlah baris: 3,132,863
Jumlah kolom: 11
Status: MEMENUHI KRITERIA TUGAS


jumlah_baris,tahun_null,bulan_null,id_provinsi_null,provinsi_null,id_pemda_null,pemda_null,no_akun_null,akun_null,anggaran_null,realisasi_null,persentase_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
3132863,0,0,0,0,0,0,0,0,0,0,0


In [3]:
contoh_data = lazy_data.head(10).collect()
contoh_data

Out[3]: 
shape: (10, 11)
┌───────┬───────┬─────────────┬──────────┬───┬────────────────┬───────────┬───────────┬────────────┐
│ tahun ┆ bulan ┆ id_provinsi ┆ provinsi ┆ … ┆ akun           ┆ anggaran  ┆ realisasi ┆ persentase │
│ ---   ┆ ---   ┆ ---         ┆ ---      ┆   ┆ ---            ┆ ---       ┆ ---       ┆ ---        │
│ i16   ┆ i8    ┆ str         ┆ str      ┆   ┆ str            ┆ f64       ┆ f64       ┆ f64        │
╞═══════╪═══════╪═════════════╪══════════╪═══╪════════════════╪═══════════╪═══════════╪════════════╡
│ 2011  ┆ 1     ┆ 01          ┆ Provinsi ┆ … ┆ Pendapatan     ┆ 7.0894e12 ┆ 7.6093e12 ┆ 107.33     │
│       ┆       ┆             ┆ Aceh     ┆   ┆                ┆           ┆           ┆            │
│ 2011  ┆ 1     ┆ 01          ┆ Provinsi ┆ … ┆ PAD            ┆ 7.9728e11 ┆ 8.0284e11 ┆ 100.7      │
│       ┆       ┆             ┆ Aceh     ┆   ┆                ┆           ┆           ┆            │
│ 2011  ┆ 1     ┆ 01          ┆ Provinsi ┆ … ┆ Pajak Daerah   ┆ 6.

## Profil DuckDB

DuckDB membaca Parquet langsung dari disk. `SUMMARIZE` menghasilkan ringkasan
statistik setiap kolom, termasuk minimum, maksimum, rata-rata, dan persentase null.
Hasil dikonversi melalui Arrow menjadi Polars DataFrame agar mendukung perbedaan
versi API DuckDB.

In [4]:
connection = duckdb.connect()
try:
    result = connection.execute(
        'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
    )
    tabel = (result.to_arrow_table() if hasattr(result, 'to_arrow_table')
             else result.fetch_arrow_table())
    profil_duckdb = pl.from_arrow(tabel)
finally:
    connection.close()
profil_duckdb

Out[4]: 
shape: (11, 12)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬─────────┬───────────┐
│ column_nam ┆ column_ty ┆ min       ┆ max       ┆ … ┆ q50       ┆ q75       ┆ count   ┆ null_perc │
│ e          ┆ pe        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---     ┆ entage    │
│ ---        ┆ ---       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ i64     ┆ ---       │
│ str        ┆ str       ┆           ┆           ┆   ┆           ┆           ┆         ┆ decimal[9 │
│            ┆           ┆           ┆           ┆   ┆           ┆           ┆         ┆ ,2]       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═════════╪═══════════╡
│ tahun      ┆ SMALLINT  ┆ 2011      ┆ 2026      ┆ … ┆ 2019      ┆ 2022      ┆ 3132863 ┆ 0.00      │
│ bulan      ┆ TINYINT   ┆ 1         ┆ 12        ┆ … ┆ 6         ┆ 9         ┆ 3132863 ┆ 0.00      │
│ id_provins ┆ VARCHAR   ┆ 01        ┆ 38        ┆ … ┆ null      ┆

<ipython-input-4-a58e314490ab>:6: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  tabel = (result.to_arrow_table() if hasattr(result, 'to_arrow_table')


## Catatan Milestone 1

Pemeriksaan berikut melengkapi ringkasan statistik dengan jumlah nilai negatif,
rasio di luar rentang eksploratif 0–200%, rasio null, anggaran nol, serta duplikasi
kunci laporan. Nilai ekstrem adalah indikasi untuk ditinjau, bukan bukti kesalahan.

In [5]:
kunci_bisnis = ['tahun', 'bulan', 'id_provinsi', 'id_pemda', 'no_akun']
anomali = lazy_data.select(
    (pl.col('anggaran') < 0).sum().alias('anggaran_negatif'),
    (pl.col('realisasi') < 0).sum().alias('realisasi_negatif'),
    (pl.col('anggaran') == 0).sum().alias('anggaran_nol'),
    (pl.col('persentase') < 0).sum().alias('persentase_negatif'),
    (pl.col('persentase') > 200).sum().alias('persentase_di_atas_200'),
    pl.col('persentase').is_null().sum().alias('persentase_null'),
    (~pl.col('bulan').is_between(1, 12)).sum().alias('bulan_tidak_valid'),
    (pl.len() - pl.struct(kunci_bisnis).n_unique()).alias('duplikat_kunci'),
).collect()
anomali

Out[5]: 
shape: (1, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ anggaran_n ┆ realisasi_ ┆ anggaran_n ┆ persentas ┆ persentas ┆ persentas ┆ bulan_tid ┆ duplikat_ │
│ egatif     ┆ negatif    ┆ ol         ┆ e_negatif ┆ e_di_atas ┆ e_null    ┆ ak_valid  ┆ kunci     │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ _200      ┆ ---       ┆ ---       ┆ ---       │
│ u32        ┆ u32        ┆ u32        ┆ u32       ┆ ---       ┆ u32       ┆ u32       ┆ u32       │
│            ┆            ┆            ┆           ┆ u32       ┆           ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 75086      ┆ 75550      ┆ 272496     ┆ 5443      ┆ 157764    ┆ 0         ┆ 0         ┆ 0         │
└────────────┴────────────┴────────────┴───────────┴───────────┴───────────┴───────────┴───────────┘


### Temuan awal

1. Dataset berukuran **14.24 MB** dengan **3,132,863 baris** dan **11 kolom**. Kriteria tugas dipenuhi melalui jumlah baris yang melebihi 1 juta.
2. Schema mencakup tahun dan bulan bertipe integer, identitas/nama wilayah dan akun bertipe string atau integer, serta anggaran, realisasi, dan persentase bertipe floating point. Kolom `tahun` dan `bulan` mendukung analisis temporal tanpa timestamp lengkap.
3. Tidak ditemukan null pada seluruh kolom.
4. Ditemukan **75,086** nilai anggaran negatif, **75,550** realisasi negatif, dan **272,496** anggaran nol. Nilai ini perlu diperiksa pada cleaning; anggaran nol memerlukan aturan rasio yang jelas.
5. Terdapat **5,443** persentase negatif, **157,764** persentase di atas 200%, serta **0** persentase null. Periksa definisi persentase dan kesesuaian dengan nominal sebelum menetapkan penanganannya.
6. Ditemukan **0** duplikat berdasarkan kunci tahun–bulan–provinsi–pemda–nomor akun dan **0** nilai bulan di luar 1–12.

Akun APBD bersifat hierarkis. Agregasi pada tahap berikutnya harus memisahkan akun induk dan anak agar nilai tidak terhitung ganda. Nominal yang besar belum tentu anomali karena skala keuangan tiap pemda berbeda.
